# **Les 2 - Lineaire Regressie**

**Doel:** Voorspel de prijs van tweedehands auto's op basis van enkele kenmerken.
    
***Stel jezelf de volgende vragen?***  
*- Welke parameter tracht je te voorspellen?*  
*- Wat voor type parameter is dit?*  
*- Wat zou een goede techniek zijn om deze te analyseren?*

## Stap 1: Libraries

In [51]:
import re
import numpy as np
import pandas as pd
# import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import joblib


## Stap 2: Data

In [52]:
# Laden
df1 = pd.read_csv("second_hand_cars_deel1.csv")
df2 = pd.read_csv("second_hand_cars_deel2.csv")

# Chassisnummer gelijk benoemen en datasets samenvoegen
df1 = df1.rename(columns={"CHASSISNUMMER": "chassisnummer"})
df = pd.merge(df1, df2, on="chassisnummer")

# Tekst omzetten naar getallen
def naar_getal(waarde):
    if pd.isna(waarde):
        return np.nan

    tekst = str(waarde).replace(".", "")
    getallen = re.findall(r"\d+(?:,\d+)?", tekst)

    if len(getallen) == 0:
        return np.nan

    getallen = [float(x.replace(",", ".")) for x in getallen]
    return np.mean(getallen)

numerieke_kolommen = [
    "CC/Batterij",
    "Oorspronkelijke Catalogusprijs",
    "Huidige Nieuwprijs",
    "Leeftijd",
    "Kilometerstand",
    "Verbruik",
    "Topsnelheid",
    "Vermogen",
    "Koppel",
    "Huidige Verkoopprijs"
]

for kolom in numerieke_kolommen:
    df[kolom] = df[kolom].apply(naar_getal)

# Duidelijke foutieve verkoopprijs verwijderen
df.loc[df["Huidige Verkoopprijs"] > 100000, "Huidige Verkoopprijs"] = np.nan

# EDA
print(df.shape)
display(df.head())
display(df.describe())


(1000, 16)


,chassisnummer,Company Naam,Auto Namen,Motor,CC/Batterij,Oorspronkelijke Catalogusprijs,Huidige Nieuwprijs,Leeftijd,Kilometerstand,Score,Conditie,Verbruik,Topsnelheid,Vermogen,Koppel,Huidige Verkoopprijs
0,1HGYAKH1Z6C000000,VOLKSWAGEN,polo,1.0 TSI,999.0,18748.0,27937.0,3.0,78945.0,1,2,14.0,177.0,73.0,123.0,12296.0
1,WAUPAHTAGYU000001,Volkswagen,Golf,1.6 TDI,1598.0,20717.0,30137.0,6.0,NaN,5,9,9.0,148.0,NaN,95.0,9975.0
2,WBAHW3A30MF000002,toyota,YARIS,1.5 hybrid,1490.0,24045.0,26977.0,2.0,132538.0,2,8,NaN,181.0,53.0,97.0,7539.0
3,VF18MDD4W31000003,Toyota,COROLLA,1.8 hybrid,1798.0,20090.0,25283.0,4.0,101065.0,4,3,11.0,197.0,54.0,116.0,8550.0
4,ZARU9NV4X5W000004,Ford,fiesta,1.0 ECOBOOST,998.0,24199.0,28397.0,6.0,61559.0,3,9,12.0,160.0,53.0,105.0,18589.0


,CC/Batterij,Oorspronkelijke Catalogusprijs,Huidige Nieuwprijs,Leeftijd,Kilometerstand,Score,Conditie,Verbruik,Topsnelheid,Vermogen,Koppel,Huidige Verkoopprijs
count,946.000000,963.000000,972.000000,1000.000000,9.770000e+02,1000.000000,1000.000000,953.000000,946.000000,968.000000,937.000000,971.000000
mean,1272.413319,21050.933541,27982.431070,4.561000,1.127825e+05,2.988000,5.592000,11.625393,166.935518,84.798554,103.801494,10727.224511
std,270.850640,2044.307080,1993.821944,1.719079,3.977857e+05,1.402791,2.824449,2.226951,19.380049,20.545903,20.959237,4427.376040
min,898.000000,17509.000000,24501.000000,2.000000,3.500000e+03,1.000000,1.000000,8.000000,135.000000,50.000000,68.000000,0.000000
25%,999.000000,19203.500000,26304.500000,3.000000,7.432100e+04,2.000000,3.000000,10.000000,150.000000,67.000000,85.000000,7210.000000
50%,1199.000000,21038.000000,27941.000000,5.000000,9.934000e+04,3.000000,6.000000,12.000000,166.000000,84.500000,104.000000,10672.000000
75%,1499.000000,22832.500000,29674.500000,6.000000,1.250450e+05,4.000000,8.000000,13.000000,184.000000,102.000000,122.000000,14467.500000
max,1798.000000,24495.000000,31493.000000,7.000000,1.250000e+07,5.000000,10.000000,15.000000,200.000000,120.000000,140.000000,20449.000000


## Stap 3: Preprocessing

In [53]:
# Numerieke kolommen
num_features = [
    "CC/Batterij",
    "Oorspronkelijke Catalogusprijs",
    "Huidige Nieuwprijs",
    "Leeftijd",
    "Kilometerstand",
    "Score",
    "Conditie",
    "Verbruik",
    "Topsnelheid",
    "Vermogen",
    "Koppel"
]

# Categorische kolommen
cat_features = [
    "Company Naam",
    "Auto Namen",
    "Motor"
]

# Preprocessing numerieke & categorische data
num_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

cat_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

# In 1 preprocessor
preprocessor = ColumnTransformer([
    ("num", num_transformer, num_features),
    ("cat", cat_transformer, cat_features)
])


## Stap 4: Splitsen in Test en Training

In [54]:
df = df.dropna(subset=["Huidige Verkoopprijs"])

X = df[num_features + cat_features]
y = df["Huidige Verkoopprijs"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


## Stap 5: ML Pipeline

In [55]:
pipeline = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])


## Stap 6: Train het model

In [56]:
pipeline.fit(X_train, y_train)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('preprocessor', ...), ('regressor', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: float, default=0.3If the output of the different transformer

## Stap 7: Evaluatie

In [57]:
y_pred = pipeline.predict(X_test)

mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("MSE:", mse)
print("RMSE:", rmse)
print("R²:", r2)


MSE: 22288663.24711781
RMSE: 4721.087083195755
R²: -0.1564046586332033


### *Intermezzo: Sla het model op*

In [58]:
# Sla de pipeline op en laad deze opnieuw
joblib.dump(pipeline, "linear_regression_pipeline.pkl")
loaded_pipeline = joblib.load("linear_regression_pipeline.pkl")


## Stap 8: Voorspelling

In [59]:
nieuwe_auto = pd.DataFrame([{
    "CC/Batterij": 1499,
    "Oorspronkelijke Catalogusprijs": 24000,
    "Huidige Nieuwprijs": 28000,
    "Leeftijd": 5,
    "Kilometerstand": 65000,
    "Score": 4,
    "Conditie": 8,
    "Verbruik": 14,
    "Topsnelheid": 190,
    "Vermogen": 150,
    "Koppel": 220,
    "Company Naam": "Volkswagen",
    "Auto Namen": "Golf",
    "Motor": "1.5 hybrid"
}])

nieuwe_auto


,CC/Batterij,Oorspronkelijke Catalogusprijs,Huidige Nieuwprijs,Leeftijd,Kilometerstand,Score,Conditie,Verbruik,Topsnelheid,Vermogen,Koppel,Company Naam,Auto Namen,Motor
0,1499,24000,28000,5,65000,4,8,14,190,150,220,Volkswagen,Golf,1.5 hybrid


In [60]:
geschatte_prijs = loaded_pipeline.predict(nieuwe_auto)

print(f"Geschatte verkoopprijs voor de testauto: € {geschatte_prijs[0]:,.2f}")


Geschatte verkoopprijs voor de testauto: € 12,883.75
